# CodePause — Colab T4 QLoRA Adapter Probe

**Phase**: 0.8  
**Hardware**: Google Colab T4 (PCIe 15GB)  
**Goal**: Validate the 4-bit QLoRA pipeline (`training/sft_lora.py --load_in_4bit`) on real T4 hardware before AMD MI300X credit is approved.

---

### What this notebook does

1. Mount Google Drive so adapter/checkpoints survive session disconnection
2. Verify T4 GPU is present (hard stop if missing)
3. Install dependencies (`bitsandbytes`, `peft`, `trl`, `transformers`)
4. Clone the CodePause repo (or pull latest)
5. Run smoke training: `sft_lora.py --smoke --load_in_4bit --max_steps 10 --limit_samples 8`
6. Evaluate the adapter with mock evaluation
7. Persist adapter + evaluation results to Drive

**Expected runtime**: ~10-15 minutes for the probe cell.

### Hard stops

| Condition | Action |
|-----------|--------|
| T4 not detected | Do not proceed — stop before model load |
| OOM on model load | Reduce `max_seq_length` or switch to TinyLlama preset |
| Training loss NaN | Abort and report in results |
| Mock pass rate < 50% | Flag for investigation before MI300X run |

### Output artifacts

Saved to `MyDrive/codepause/colab-t4-qlora/`:
- `adapter/` — LoRA adapter files (adapter_model.safetensors, adapter_config.json)
- `results/` — evaluation JSONL and report
- `logs/training_log.txt` — captured training stdout

---

## 1. Drive Mount & Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_BASE = '/content/drive/MyDrive/codepause/colab-t4-qlora'
ADAPTER_DIR = f'{OUTPUT_BASE}/adapter'
RESULTS_DIR = f'{OUTPUT_BASE}/results'
LOGS_DIR = f'{OUTPUT_BASE}/logs'

import os
for d in [OUTPUT_BASE, ADAPTER_DIR, RESULTS_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Output base: {OUTPUT_BASE}')


import subprocess, sys, time


def run_step(name, cmd, cwd=None, log_path=None):
    print(f"\n=== {name} - START ===", flush=True)
    print(" ".join(cmd), flush=True)
    start = time.time()
    output_lines = []
    log_handle = open(log_path, "w", encoding="utf-8") if log_path else None
    try:
        proc = subprocess.Popen(
            cmd,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            output_lines.append(line)
            print(line, end="", flush=True)
            sys.stdout.flush()
            if log_handle:
                log_handle.write(line)
                log_handle.flush()
        return_code = proc.wait()
    finally:
        if log_handle:
            log_handle.close()
    elapsed = time.time() - start
    print(f"\n=== {name} - END ({elapsed:.1f}s, rc={return_code}) ===", flush=True)
    output = "".join(output_lines)
    if return_code != 0:
        tail = "".join(output_lines[-40:])
        raise RuntimeError(f"HARD STOP: {name} failed with exit code {return_code}\n{tail}")
    return output


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output base: /content/drive/MyDrive/codepause/colab-t4-qlora


## 2. GPU Check (HARD STOP)

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    raise RuntimeError(
        'HARD STOP: No CUDA GPU detected. '
        'This notebook requires a T4 or better GPU. '
        'Go to Runtime → Change runtime type → GPU and retry.'
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}')
print(f'GPU memory: {gpu_mem:.1f} GB')

# HARD STOP: verify it's a T4 (or at least not CPU)
if 'T4' not in gpu_name and 'A100' not in gpu_name and 'L4' not in gpu_name:
    raise RuntimeError(
        f'HARD STOP: Expected T4/A100/L4 but found "{gpu_name}". '
        'Only Pascal+ GPUs with bitsandbytes support are valid for QLoRA.'
    )

print('GPU check PASSED — proceeding to install.')

PyTorch version: 2.10.0+cpu
CUDA available: False


RuntimeError: HARD STOP: No CUDA GPU detected. This notebook requires a T4 or better GPU. Go to Runtime → Change runtime type → GPU and retry.

## 3. Install Dependencies & Fix BitsAndBytes

Colab T4 with CUDA 12.8 can have `bitsandbytes` loading issues (`CUDA SETUP: CUDA detection failed`).
The solution is to purge the pip cache and force a clean reinstall from PyPI.

**Important**: If the `import bitsandbytes` step below fails, **DO NOT PROCEED**. You may need to restart the session (`Runtime -> Restart session`) and run again.

In [ ]:
# Clean old/incompatible bitsandbytes first
!pip uninstall -y bitsandbytes
!pip cache purge

# Install CUDA-compatible stack
!pip install -U --no-cache-dir bitsandbytes transformers datasets accelerate peft trl scipy pytest pytest-cov pyyaml

# Verify CUDA + bitsandbytes
import torch
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

try:
    import bitsandbytes as bnb
    print("bitsandbytes:", bnb.__version__)
except ImportError as e:
    print(e)
    raise RuntimeError("HARD STOP: bitsandbytes failed to load. Do not run training. Try restarting the runtime.")


## 4. Clone CodePause + Required Path Gate

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/alesierraalta/AMD-Hacktaton-thinking-middle.git'
PROJECT_ROOT = Path('/content/codepause-amd')
REPO_DIR = str(PROJECT_ROOT)

%cd /content
!rm -rf /content/codepause-amd
!git clone {REPO_URL} /content/codepause-amd
%cd /content/codepause-amd

!git branch --show-current
!git log --oneline -5
!find /content/codepause-amd -maxdepth 4 -type f -name 'sft_lora.py'
!ls -la /content/codepause-amd/training

SFT_SCRIPT = PROJECT_ROOT / 'training' / 'sft_lora.py'
DATASET_PATH = PROJECT_ROOT / 'data' / 'thinkanywhere_sft.jsonl'
required_paths = [PROJECT_ROOT, SFT_SCRIPT, DATASET_PATH]

for path in required_paths:
    print(path, 'exists:', path.exists())
    if not path.exists():
        raise FileNotFoundError(f'Required path not found: {path}')

print('Repo OK:', PROJECT_ROOT)
print('SFT script OK:', SFT_SCRIPT)
print('Dataset OK:', DATASET_PATH)

## 5. T4 QLoRA Training Probe (HARD STOP on OOM)

In [ ]:
import os

TRAINING_CMD = [
    'python', f'{REPO_DIR}/training/sft_lora.py',
    '--model_name', 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    '--dataset_path', f'{REPO_DIR}/data/thinkanywhere_sft.jsonl',
    '--output_dir', ADAPTER_DIR,
    '--load_in_4bit',
    '--max_steps', '10',
    '--limit_samples', '8',
    '--per_device_train_batch_size', '1',
]

training_output = run_step(
    'T4 QLoRA training probe',
    TRAINING_CMD,
    log_path=f'{LOGS_DIR}/training_log.txt',
)

if 'CUDA out of memory' in training_output or 'OutOfMemoryError' in training_output:
    raise RuntimeError(
        'HARD STOP: T4 OOM during QLoRA probe. '
        'Reduce max_steps/limit_samples or switch to the tiny preset.'
    )

if not os.path.exists(ADAPTER_DIR):
    raise RuntimeError(f'HARD STOP: Adapter output dir missing: {ADAPTER_DIR}')

print('Training probe completed.')


## 6. Validate Adapter Structure

In [ ]:
import os, json

adapter_path = f'{ADAPTER_DIR}'
config_file = f'{adapter_path}/adapter_config.json'

if not os.path.exists(config_file):
    raise FileNotFoundError(
        f'HARD STOP: adapter_config.json not found at {config_file}. '
        'Training may have failed silently.'
    )

with open(config_file) as f:
    config = json.load(f)

print(f'Adapter config: {json.dumps(config, indent=2)}')

# Verify expected keys
required_keys = ['base_model_name_or_path', 'peft_type', 'task_type']
for key in required_keys:
    if key not in config:
        raise ValueError(f'Missing required config key: {key}')

# Check adapter_model.safetensors exists
adapter_weights = f'{adapter_path}/adapter_model.safetensors'
if not os.path.exists(adapter_weights):
    raise FileNotFoundError(f'HARD STOP: adapter weights not found at {adapter_weights}')

weight_size_gb = os.path.getsize(adapter_weights) / 1e9
print(f'Adapter weights: {weight_size_gb:.2f} GB')

print('Adapter structure VALID.')

## 7. Mock Evaluation with Metadata Injection

In [ ]:
import os, json

METADATA_PATH = f'{RESULTS_DIR}/metadata_colab_t4.json'
METADATA = {
    'platform': 'colab-t4',
    'quantization': '4bit-nf4',
    'model_name': 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    'colab_runtime': 'T4',
}
with open(METADATA_PATH, 'w') as f:
    json.dump(METADATA, f)

EVAL_CMD = [
    'python', f'{REPO_DIR}/eval/evaluate_finetuned.py',
    '--base_model', 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    '--adapter_path', ADAPTER_DIR,
    '--problems_path', f'{REPO_DIR}/data/problems.jsonl',
    '--output_path', f'{RESULTS_DIR}/finetuned_colab_t4.jsonl',
    '--mock',
    '--metadata_json', METADATA_PATH,
]

run_step('Mock fine-tuned evaluation with metadata', EVAL_CMD)

if not os.path.exists(f'{RESULTS_DIR}/finetuned_colab_t4.jsonl'):
    raise RuntimeError('HARD STOP: Fine-tuned evaluation output missing.')

print('Mock evaluation completed.')


## 8. Generate Report

In [ ]:
import os, json

BASELINE_CMD = [
    'python', f'{REPO_DIR}/eval/evaluate_baseline.py',
    '--problems_path', f'{REPO_DIR}/data/problems.jsonl',
    '--output_path', f'{RESULTS_DIR}/baseline_mock.jsonl',
    '--mock',
]
run_step('Baseline mock evaluation', BASELINE_CMD)

REPORT_CMD = [
    'python', f'{REPO_DIR}/eval/make_report.py',
    '--baseline', f'{RESULTS_DIR}/baseline_mock.jsonl',
    '--finetuned', f'{RESULTS_DIR}/finetuned_colab_t4.jsonl',
    '--out', f'{RESULTS_DIR}/colab_t4_report.md',
]
run_step('Generate Colab T4 comparison report', REPORT_CMD)

if not os.path.exists(f'{RESULTS_DIR}/colab_t4_report.md'):
    raise RuntimeError('HARD STOP: Report generation failed.')

print(f'Report written: {RESULTS_DIR}/colab_t4_report.md')


## 9. Copy Outputs to Drive

In [ ]:
import shutil, os

# Verify all critical outputs exist
checkpoints = [
    f'{ADAPTER_DIR}/adapter_config.json',
    f'{ADAPTER_DIR}/adapter_model.safetensors',
    f'{RESULTS_DIR}/finetuned_colab_t4.jsonl',
    f'{RESULTS_DIR}/colab_t4_report.md',
    f'{LOGS_DIR}/training_log.txt',
]

all_ok = True
for path in checkpoints:
    status = '✓' if os.path.exists(path) else '✗ MISSING'
    print(f'{status}  {path}')
    if not os.path.exists(path):
        all_ok = False

if not all_ok:
    raise RuntimeError('HARD STOP: Some outputs are missing. Do not destroy this session.')

# Persist a session metadata file
import json
session_meta = {
    'platform': 'colab-t4',
    'gpu': gpu_name,
    'status': 'probe_complete',
    'adapter_path': ADAPTER_DIR,
    'results_dir': RESULTS_DIR,
}
with open(f'{OUTPUT_BASE}/session_meta.json', 'w') as f:
    json.dump(session_meta, f, indent=2)

print('\nAll outputs verified and persisted to Drive.')
print(f'Adapter: {ADAPTER_DIR}')
print(f'Results: {RESULTS_DIR}')
print('Colab T4 QLoRA probe COMPLETE.')